# Multi-Dataset Cardiac Biomarker Prediction Pipeline
### Datasets Covered:
1. **Zheen Hospital Heart Attack Dataset** (Troponin, CK-MB, Blood Sugar, Vitals)
2. **UCI Heart Failure Clinical Records** (Serum Creatinine, CPK, Serum Sodium, Platelets, Ejection Fraction)
3. **Framingham Heart Study** (Total Cholesterol, Glucose, BP, BMI, Heart Rate -> 10-Year CHD)
4. **UCI Myocardial Infarction (MI) Complications** (AST/ALT, K+, Na+, WBC, ESR, ICU Vitals -> Mortality & Complications)

### Best Model Architecture for Clinical Biomarker Tabular Data:
- **Primary Models:** Gradient Boosted Decision Trees (**CatBoost**, **XGBoost**, **LightGBM**) + **Stacked Ensemble** (Logistic Regression meta-learner).
- **Baselines:** Logistic Regression & Random Forest.
- **Explainability:** **SHAP (TreeExplainer)** to quantify individual biomarker contributions.
- **Leakage Prevention:** All imputation, scaling, and SMOTE/class-weighting happen strictly inside Stratified 5-Fold Cross-Validation pipelines.


In [1]:
# Uncomment and run this cell once if any library is missing in your environment
# !pip install -q pandas numpy scikit-learn xgboost lightgbm catboost imbalanced-learn shap matplotlib seaborn joblib


In [2]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, accuracy_score,
    matthews_corrcoef, roc_curve, precision_recall_curve
)

# Optional high-performance GBDT & XAI libraries
HAS_XGB = HAS_LGBM = HAS_CAT = HAS_SMOTE = HAS_SHAP = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print("[Info] xgboost not installed. Install via: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    print("[Info] lightgbm not installed. Install via: pip install lightgbm")

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    print("[Info] catboost not installed. Install via: pip install catboost")

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_SMOTE = True
except ImportError:
    print("[Info] imbalanced-learn not installed. Using class_weight balancing.")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    print("[Info] shap not installed. Install via: pip install shap")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
print(f"Loaded | XGBoost: {HAS_XGB} | LightGBM: {HAS_LGBM} | CatBoost: {HAS_CAT} | SMOTE: {HAS_SMOTE} | SHAP: {HAS_SHAP}")


Loaded | XGBoost: True | LightGBM: True | CatBoost: True | SMOTE: True | SHAP: True


In [3]:
current = Path.cwd()
PROJECT_ROOT = None
for candidate in [current] + list(current.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the ml-service project root.")

# ==============================================================================
# 1. CONFIGURATION: SET YOUR LOCAL FILE PATHS HERE
# ==============================================================================
DATA_PATHS = {
    "Zheen": PROJECT_ROOT / "data" / "raw" / "biomarkers" / "zheen" / "Medicaldataset.csv",
    "UCI_Heart_Failure": PROJECT_ROOT / "data" / "raw" / "biomarkers" / "uci_heart_failure" / "Heart_failure_clinical_records_dataset.csv",
    "Framingham": PROJECT_ROOT / "data" / "raw" / "biomarkers" / "framingham" / "framingham.csv",
    "MI_Complications": PROJECT_ROOT / "data" / "raw" / "biomarkers" / "mi_complications" / "Myocardial infarction complications Database.csv",
}

# Clinical pipeline options
INCLUDE_UCI_FOLLOWUP_TIME = False   # False avoids follow-up time leakage (biomarker-only baseline prognosis)
MI_TARGET_COL = "LET_IS"            # 'LET_IS' (mortality > 0) or any complication e.g. 'ZSN', 'FIBR_PREDS', 'OTEK_LANC'
USE_SMOTE_FOR_IMBALANCED = True     # Applies SMOTE inside CV folds when minority class < 25%
N_SPLITS = 5
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "metrics" / "biomarkers" / "cardiac_biomarker_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [4]:
# ==============================================================================
# 2. DATASET LOADERS & CLINICAL BIOMARKER FEATURE ENGINEERING
# ==============================================================================

def _normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]
    return df


def load_and_engineer_zheen(path: str):
    """
    Zheen Hospital Heart Attack Dataset (1,319 records).
    Core Biomarkers: Troponin, CK-MB, Blood Sugar + Hemodynamics.
    """
    df = _normalize_cols(pd.read_csv(path))

    target_col = "result" if "result" in df.columns else df.columns[-1]
    if df[target_col].dtype == object:
        y = df[target_col].astype(str).str.strip().str.lower().map({"positive": 1, "negative": 0, "1": 1, "0": 0})
    else:
        y = df[target_col].astype(int)

    X = df.drop(columns=[target_col]).copy()

    hr_col = next((c for c in X.columns if "heart" in c or c == "impluse"), None)
    sbp_col = next((c for c in X.columns if "systolic" in c or "pressurehight" in c), None)
    dbp_col = next((c for c in X.columns if "diastolic" in c or "pressurelow" in c), None)
    sugar_col = next((c for c in X.columns if "sugar" in c or "glucose" in c), None)
    ckmb_col = next((c for c in X.columns if "ck" in c or "kcm" in c), None)
    trop_col = next((c for c in X.columns if "troponin" in c), None)

    # Clean known extreme outlier in Zheen heart_rate (e.g., 1111 bpm)
    if hr_col:
        X[hr_col] = X[hr_col].clip(lower=30, upper=240)

    # Biomarker & Hemodynamic Feature Engineering
    if sbp_col and dbp_col:
        X["pulse_pressure"] = X[sbp_col] - X[dbp_col]
        X["mean_arterial_pressure"] = X[dbp_col] + (X["pulse_pressure"] / 3.0)
    if hr_col and sbp_col:
        X["shock_index"] = X[hr_col] / (X[sbp_col].replace(0, np.nan))
    if trop_col:
        X["log_troponin"] = np.log1p(X[trop_col].clip(lower=0))
        X["troponin_elevated_flag"] = (X[trop_col] > 0.04).astype(int)
    if ckmb_col:
        X["log_ckmb"] = np.log1p(X[ckmb_col].clip(lower=0))
        X["ckmb_elevated_flag"] = (X[ckmb_col] > 25.0).astype(int)
    if trop_col and ckmb_col:
        X["myocardial_injury_index"] = X["log_troponin"] * X["log_ckmb"]
    if sugar_col:
        X["hyperglycemia_flag"] = (X[sugar_col] > 140).astype(int)

    biomarkers = [c for c in [trop_col, ckmb_col, sugar_col, "log_troponin", "log_ckmb", "myocardial_injury_index"] if c in X.columns]
    return X, y.astype(int), biomarkers


def load_and_engineer_uci_hf(path: str, include_time: bool = False):
    """
    UCI Heart Failure Clinical Records (299 records).
    Core Biomarkers: Serum Creatinine, Ejection Fraction, CPK, Serum Sodium, Platelets.
    """
    df = _normalize_cols(pd.read_csv(path))
    target_col = "death_event" if "death_event" in df.columns else df.columns[-1]
    y = df[target_col].astype(int)
    X = df.drop(columns=[target_col]).copy()

    if not include_time and "time" in X.columns:
        X = X.drop(columns=["time"])

    # Biomarker Feature Engineering (Chicco & Jurman 2020 inspired)
    if "creatinine_phosphokinase" in X.columns:
        X["log_cpk"] = np.log1p(X["creatinine_phosphokinase"].clip(lower=0))
    if "serum_creatinine" in X.columns and "ejection_fraction" in X.columns:
        X["creatinine_to_ef_ratio"] = X["serum_creatinine"] / (X["ejection_fraction"] + 1e-5)
        X["cardiorenal_risk_flag"] = ((X["serum_creatinine"] > 1.5) & (X["ejection_fraction"] < 40)).astype(int)
    if "serum_sodium" in X.columns and "serum_creatinine" in X.columns:
        X["sodium_to_creatinine_ratio"] = X["serum_sodium"] / (X["serum_creatinine"] + 1e-5)
        X["hyponatremia_flag"] = (X["serum_sodium"] < 135).astype(int)
    if "platelets" in X.columns:
        X["log_platelets"] = np.log1p(X["platelets"].clip(lower=0))

    biomarkers = [c for c in [
        "serum_creatinine", "ejection_fraction", "creatinine_phosphokinase",
        "serum_sodium", "platelets", "creatinine_to_ef_ratio", "sodium_to_creatinine_ratio"
    ] if c in X.columns]
    return X, y, biomarkers


def load_and_engineer_framingham(path: str):
    """
    Framingham Heart Study Dataset (~4,240 records).
    Core Biomarkers: Total Cholesterol, Glucose, Systolic/Diastolic BP, BMI, Heart Rate.
    """
    df = _normalize_cols(pd.read_csv(path))
    target_col = "tenyearchd" if "tenyearchd" in df.columns else df.columns[-1]
    y = df[target_col].astype(int)
    X = df.drop(columns=[target_col]).copy()

    if "education" in X.columns:
        X = X.drop(columns=["education"])

    # Biomarker Feature Engineering
    if "sysbp" in X.columns and "diabp" in X.columns:
        X["pulse_pressure"] = X["sysbp"] - X["diabp"]
        X["mean_arterial_pressure"] = X["diabp"] + (X["pulse_pressure"] / 3.0)
        X["hypertension_stage2_flag"] = ((X["sysbp"] >= 140) | (X["diabp"] >= 90)).astype(int)
    if "glucose" in X.columns:
        X["log_glucose"] = np.log1p(X["glucose"].clip(lower=0))
        X["impaired_glucose_flag"] = (X["glucose"] >= 126).astype(int)
    if "totchol" in X.columns and "bmi" in X.columns:
        X["chol_bmi_product"] = X["totchol"] * X["bmi"]
    if "age" in X.columns and "sysbp" in X.columns:
        X["age_x_sysbp"] = X["age"] * X["sysbp"]

    biomarkers = [c for c in [
        "totchol", "glucose", "sysbp", "diabp", "bmi", "heartrate",
        "pulse_pressure", "mean_arterial_pressure", "log_glucose"
    ] if c in X.columns]
    return X, y, biomarkers


def load_and_engineer_mi_complications(path: str, target_col: str = "LET_IS"):
    """
    UCI Myocardial Infarction Complications Dataset (1,700 records, 124 columns).
    Core Blood Biomarkers: K_BLOOD, NA_BLOOD, ALT_BLOOD, AST_BLOOD, L_BLOOD (WBC), ROE (ESR).
    """
    df = pd.read_csv(path, na_values=["?", "NA", "nan", ""])
    df = _normalize_cols(df)

    for id_col in ["id", "unnamed:_0"]:
        if id_col in df.columns:
            df = df.drop(columns=[id_col])

    known_outcomes = [
        "fibr_preds", "preds_tah", "jelud_tah", "fibr_jelud", "a_v_blok",
        "otek_lanc", "razriv", "dressler", "zsn", "rec_im", "p_im_sten", "let_is"
    ]
    t_col = target_col.strip().lower()
    if t_col not in df.columns:
        t_col = df.columns[-1]

    y = (pd.to_numeric(df[t_col], errors="coerce").fillna(0) > 0).astype(int)

    # Drop ALL 12 outcome columns from X to prevent target leakage
    outcome_cols_present = [c for c in known_outcomes if c in df.columns]
    if t_col not in outcome_cols_present:
        outcome_cols_present.append(t_col)
    X = df.drop(columns=outcome_cols_present).copy()

    X = X.apply(pd.to_numeric, errors="coerce")

    # Drop columns with >50% missing values (e.g., kfk_blood is >99% missing in UCI MI)
    missing_frac = X.isna().mean()
    X = X.loc[:, missing_frac <= 0.50].copy()

    # Clinical Blood Biomarker Engineering
    if "ast_blood" in X.columns and "alt_blood" in X.columns:
        X["ast_alt_ratio"] = X["ast_blood"] / (X["alt_blood"].replace(0, np.nan) + 1e-5)
    if "na_blood" in X.columns and "k_blood" in X.columns:
        X["na_k_ratio"] = X["na_blood"] / (X["k_blood"].replace(0, np.nan) + 1e-5)
        X["hypokalemia_flag"] = (X["k_blood"] < 3.5).astype(int)
    if "s_ad_orit" in X.columns and "d_ad_orit" in X.columns:
        X["icu_pulse_pressure"] = X["s_ad_orit"] - X["d_ad_orit"]
    if "l_blood" in X.columns:
        X["log_wbc"] = np.log1p(X["l_blood"].clip(lower=0))

    biomarkers = [c for c in [
        "k_blood", "na_blood", "alt_blood", "ast_blood", "l_blood", "roe",
        "ast_alt_ratio", "na_k_ratio", "icu_pulse_pressure", "log_wbc"
    ] if c in X.columns]
    return X, y, biomarkers


In [5]:
# ==============================================================================
# 3. MODEL FACTORY (GBDTs + STACKED ENSEMBLE + BASELINES)
# ==============================================================================

def get_models(pos_weight: float = 1.0):
    """Returns tuned clinical tabular classifiers + stacked biomarker ensemble."""
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000, class_weight="balanced", C=0.5, random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=8, min_samples_leaf=3,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=250, max_depth=5, learning_rate=0.05,
            l2_regularization=1.5, class_weight="balanced", random_state=RANDOM_STATE
        ),
    }

    stack_estimators = []

    if HAS_XGB:
        xgb = XGBClassifier(
            n_estimators=250, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5,
            scale_pos_weight=pos_weight, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )
        models["XGBoost"] = xgb
        stack_estimators.append(("xgb", xgb))

    if HAS_LGBM:
        lgbm = LGBMClassifier(
            n_estimators=250, max_depth=5, num_leaves=20, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.5,
            class_weight="balanced", random_state=RANDOM_STATE, verbose=-1, n_jobs=-1
        )
        models["LightGBM"] = lgbm
        stack_estimators.append(("lgbm", lgbm))

    if HAS_CAT:
        cat = CatBoostClassifier(
            iterations=350, depth=5, learning_rate=0.05, l2_leaf_reg=3.0,
            auto_class_weights="Balanced", random_seed=RANDOM_STATE, verbose=0
        )
        models["CatBoost"] = cat
        stack_estimators.append(("cat", cat))

    if len(stack_estimators) < 2:
        stack_estimators.append(("rf", models["RandomForest"]))
        stack_estimators.append(("hgb", models["HistGradientBoosting"]))

    models["Stacked_GBDT_Ensemble"] = StackingClassifier(
        estimators=stack_estimators,
        final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
        cv=3,
        stack_method="predict_proba",
        n_jobs=-1
    )
    return models


def build_pipeline(clf, needs_scaling: bool, use_smote: bool):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if needs_scaling:
        steps.append(("scaler", StandardScaler()))
    if use_smote and HAS_SMOTE:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
        steps.append(("clf", clf))
        return ImbPipeline(steps)
    steps.append(("clf", clf))
    return Pipeline(steps)


def evaluate_dataset(dataset_name: str, X: pd.DataFrame, y: pd.Series, n_splits: int = 5):
    """Runs Stratified K-Fold CV across all models and returns metrics + best fitted model."""
    pos_rate = y.mean()
    neg_count, pos_count = (y == 0).sum(), (y == 1).sum()
    pos_weight = float(neg_count) / max(float(pos_count), 1.0)
    apply_smote = USE_SMOTE_FOR_IMBALANCED and HAS_SMOTE and (pos_rate < 0.25 or pos_rate > 0.75)

    print(f"\n{'='*75}")
    print(f"DATASET: {dataset_name} | Shape: {X.shape} | Positive Rate: {pos_rate:.2%} | SMOTE: {apply_smote}")
    print(f"{'='*75}")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    models = get_models(pos_weight=1.0 if apply_smote else pos_weight)

    results = []
    oof_probas = {}

    for model_name, clf in models.items():
        oof_pred_proba = np.zeros(len(y))
        needs_scaling = model_name in ["LogisticRegression"]

        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
            y_tr = y.iloc[train_idx]

            pipe = build_pipeline(clf, needs_scaling=needs_scaling, use_smote=apply_smote)
            pipe.fit(X_tr, y_tr)
            oof_pred_proba[val_idx] = pipe.predict_proba(X_va)[:, 1]

        oof_probas[model_name] = oof_pred_proba
        y_pred = (oof_pred_proba >= 0.5).astype(int)

        metrics = {
            "Dataset": dataset_name,
            "Model": model_name,
            "ROC_AUC": roc_auc_score(y, oof_pred_proba),
            "PR_AUC": average_precision_score(y, oof_pred_proba),
            "F1": f1_score(y, y_pred, zero_division=0),
            "Recall": recall_score(y, y_pred, zero_division=0),
            "Precision": precision_score(y, y_pred, zero_division=0),
            "Accuracy": accuracy_score(y, y_pred),
            "MCC": matthews_corrcoef(y, y_pred),
        }
        results.append(metrics)

    res_df = pd.DataFrame(results).sort_values(by="ROC_AUC", ascending=False).reset_index(drop=True)
    display_cols = ["Model", "ROC_AUC", "PR_AUC", "F1", "Recall", "Precision", "MCC", "Accuracy"]
    print(res_df[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    best_model_name = res_df.iloc[0]["Model"]
    best_pipe = build_pipeline(
        models[best_model_name],
        needs_scaling=(best_model_name == "LogisticRegression"),
        use_smote=apply_smote
    )
    best_pipe.fit(X, y)
    joblib.dump(best_pipe, os.path.join(OUTPUT_DIR, f"best_model_{dataset_name}.joblib"))

    tree_candidates = [m for m in res_df["Model"] if m in ["CatBoost", "XGBoost", "LightGBM", "RandomForest"]]
    shap_model_name = tree_candidates[0] if tree_candidates else best_model_name
    shap_pipe = build_pipeline(models[shap_model_name], needs_scaling=False, use_smote=False)
    shap_pipe.fit(X, y)

    return res_df, oof_probas, best_model_name, best_pipe, shap_model_name, shap_pipe


In [6]:
# ==============================================================================
# 5. RUN PIPELINE ACROSS ALL 4 DATASETS
# ==============================================================================

LOADERS = {
    "Zheen": lambda p: load_and_engineer_zheen(p),
    "UCI_Heart_Failure": lambda p: load_and_engineer_uci_hf(p, include_time=INCLUDE_UCI_FOLLOWUP_TIME),
    "Framingham": lambda p: load_and_engineer_framingham(p),
    "MI_Complications": lambda p: load_and_engineer_mi_complications(p, target_col=MI_TARGET_COL),
}

all_results = []
oof_prediction_frames = {}

for name, path in DATA_PATHS.items():
    if not os.path.exists(path):
        print(f"[Skipped] {name}: File not found at '{path}'. Place your CSV there and re-run.")
        continue

    X, y, biomarkers = LOADERS[name](path)
    print(f"\n[{name}] Engineered Biomarker Features ({len(biomarkers)}): {biomarkers}")

    res_df, oof_probas, best_name, best_pipe, shap_name, shap_pipe = evaluate_dataset(
        name, X, y, n_splits=N_SPLITS
    )
    all_results.append(res_df)
    oof_frame = pd.DataFrame({"true": y.astype(int).to_numpy()})
    for model_name, probabilities in oof_probas.items():
        oof_frame[model_name] = probabilities
    oof_prediction_frames[name] = oof_frame


if all_results:
    summary_df = pd.concat(all_results, ignore_index=True)
    summary_path = os.path.join(OUTPUT_DIR, "all_4_datasets_benchmark.csv")
    summary_df.to_csv(summary_path, index=False)
    for dataset_name, oof_frame in oof_prediction_frames.items():
        oof_frame.to_csv(os.path.join(OUTPUT_DIR, f"{dataset_name}_oof_predictions.csv"), index=False)
    print(f"\nSaved consolidated benchmark to: {summary_path}")

    best_per_ds = summary_df.sort_values("ROC_AUC", ascending=False).groupby("Dataset", as_index=False).first()
    print("\n=== BEST MODEL PER DATASET ===")
    print(best_per_ds[["Dataset", "Model", "ROC_AUC", "PR_AUC", "F1", "Recall", "MCC"]].to_string(index=False))
else:
    print("\nNo CSV files found yet. Update DATA_PATHS in Cell 4 with your local dataset paths and re-run!")



[Zheen] Engineered Biomarker Features (6): ['troponin', 'ck_mb', 'blood_sugar', 'log_troponin', 'log_ckmb', 'myocardial_injury_index']

DATASET: Zheen | Shape: (1319, 17) | Positive Rate: 61.41% | SMOTE: False


                Model  ROC_AUC  PR_AUC     F1  Recall  Precision    MCC  Accuracy
         RandomForest   0.9970  0.9981 0.9888  0.9827     0.9950 0.9714    0.9864
              XGBoost   0.9953  0.9972 0.9901  0.9889     0.9913 0.9744    0.9879
 HistGradientBoosting   0.9947  0.9969 0.9889  0.9864     0.9913 0.9713    0.9864
             LightGBM   0.9945  0.9960 0.9895  0.9852     0.9938 0.9729    0.9871
             CatBoost   0.9937  0.9898 0.9870  0.9827     0.9913 0.9666    0.9841
Stacked_GBDT_Ensemble   0.9926  0.9922 0.9895  0.9864     0.9925 0.9729    0.9871
   LogisticRegression   0.9168  0.9536 0.8597  0.7827     0.9534 0.7028    0.8431



[UCI_Heart_Failure] Engineered Biomarker Features (7): ['serum_creatinine', 'ejection_fraction', 'creatinine_phosphokinase', 'serum_sodium', 'platelets', 'creatinine_to_ef_ratio', 'sodium_to_creatinine_ratio']

DATASET: UCI_Heart_Failure | Shape: (299, 17) | Positive Rate: 32.11% | SMOTE: False


                Model  ROC_AUC  PR_AUC     F1  Recall  Precision    MCC  Accuracy
         RandomForest   0.7849  0.6313 0.6538  0.7083     0.6071 0.4742    0.7592
   LogisticRegression   0.7586  0.6136 0.6168  0.6875     0.5593 0.4120    0.7258
              XGBoost   0.7518  0.5870 0.5846  0.5938     0.5758 0.3838    0.7291
             LightGBM   0.7457  0.5897 0.5584  0.5729     0.5446 0.3419    0.7090
 HistGradientBoosting   0.7436  0.5730 0.6040  0.6354     0.5755 0.4038    0.7324
             CatBoost   0.7395  0.5814 0.5856  0.5521     0.6235 0.4083    0.7492
Stacked_GBDT_Ensemble   0.7315  0.5420 0.6058  0.6562     0.5625 0.4002    0.7258



[Framingham] Engineered Biomarker Features (9): ['totchol', 'glucose', 'sysbp', 'diabp', 'bmi', 'heartrate', 'pulse_pressure', 'mean_arterial_pressure', 'log_glucose']

DATASET: Framingham | Shape: (4240, 21) | Positive Rate: 15.19% | SMOTE: True


                Model  ROC_AUC  PR_AUC     F1  Recall  Precision    MCC  Accuracy
   LogisticRegression   0.7216  0.3290 0.3841  0.6770     0.2681 0.2555    0.6703
         RandomForest   0.7033  0.3070 0.3201  0.3059     0.3356 0.2052    0.8026
              XGBoost   0.6795  0.3015 0.2500  0.1879     0.3735 0.1776    0.8288
Stacked_GBDT_Ensemble   0.6761  0.2949 0.2994  0.2531     0.3663 0.2046    0.8200
             CatBoost   0.6698  0.2902 0.2298  0.1630     0.3889 0.1722    0.8340
 HistGradientBoosting   0.6694  0.2847 0.2459  0.1848     0.3673 0.1726    0.8278
             LightGBM   0.6687  0.2846 0.2434  0.1786     0.3821 0.1773    0.8314



[MI_Complications] Engineered Biomarker Features (10): ['k_blood', 'na_blood', 'alt_blood', 'ast_blood', 'l_blood', 'roe', 'ast_alt_ratio', 'na_k_ratio', 'icu_pulse_pressure', 'log_wbc']

DATASET: MI_Complications | Shape: (1700, 112) | Positive Rate: 15.94% | SMOTE: True


                Model  ROC_AUC  PR_AUC     F1  Recall  Precision    MCC  Accuracy
             CatBoost   0.8828  0.6619 0.5446  0.4391     0.7169 0.5010    0.8829
 HistGradientBoosting   0.8819  0.6655 0.5520  0.4502     0.7135 0.5062    0.8835
             LightGBM   0.8817  0.6640 0.5333  0.4280     0.7073 0.4891    0.8806
              XGBoost   0.8783  0.6557 0.5505  0.4428     0.7273 0.5086    0.8847
Stacked_GBDT_Ensemble   0.8681  0.6521 0.5727  0.4871     0.6947 0.5187    0.8841
         RandomForest   0.8568  0.6235 0.4951  0.3727     0.7372 0.4673    0.8788
   LogisticRegression   0.8354  0.5504 0.5281  0.6937     0.4263 0.4315    0.8024



Saved consolidated benchmark to: D:\Piyush\Cardio Project\CardioFusion-XAI\ml-service\artifacts\metrics\biomarkers\cardiac_biomarker_pipeline\all_4_datasets_benchmark.csv

=== BEST MODEL PER DATASET ===
          Dataset              Model  ROC_AUC   PR_AUC       F1   Recall      MCC
       Framingham LogisticRegression 0.721572 0.329034 0.384141 0.677019 0.255467
 MI_Complications           CatBoost 0.882786 0.661918 0.544622 0.439114 0.500956
UCI_Heart_Failure       RandomForest 0.784893 0.631318 0.653846 0.708333 0.474190
            Zheen       RandomForest 0.997034 0.998086 0.988820 0.982716 0.971435
